### define schema

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType,
    TimestampType, BooleanType
)

TXN_SCHEMA = StructType([
    StructField("txn_id",          StringType(),    False),
    StructField("account_id",      StringType(),    False),
    StructField("customer_id",     StringType(),    False),
    StructField("counterparty_id", StringType(),    True),
    StructField("txn_type",        StringType(),    False),
    StructField("channel",         StringType(),    False),
    StructField("amount",          DoubleType(),    False),
    StructField("currency",        StringType(),    False),
    StructField("merchant_name",   StringType(),    True),
    StructField("merchant_mcc",    StringType(),    True),
    StructField("country_code",    StringType(),    False),
    StructField("is_international",BooleanType(),   False),
    StructField("device_id",       StringType(),    True),
    StructField("ip_address",      StringType(),    True),
    StructField("event_ts",        TimestampType(), False),
    StructField("kafka_offset",    IntegerType(),   True),
    StructField("kafka_partition", IntegerType(),   True),
])

### define generator

In [0]:
import random, uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row

TXN_TYPES  = ["PURCHASE","TRANSFER","ATM_WITHDRAWAL",
               "DIRECT_DEBIT","STANDING_ORDER","REFUND"]
CHANNELS   = ["mobile_app","online_banking",
               "branch","atm","pos_terminal"]
CURRENCIES = ["GBP","USD","EUR","GBP","GBP"]  # GBP-weighted
MERCHANTS  = [
    ("Amazon",        "5999"), ("Tesco",     "5411"),
    ("Shell",         "5541"), ("Unknown",   "9999"),
    ("Deliveroo",     "5812"), ("Netflix",   "7841"),
    ("HMRC",          "9311"), ("Barclays",  "6012"),
]
COUNTRIES  = ["GB","GB","GB","US","EU","NG","CN","AE"]

def generate_batch(n=500, fraud_rate=0.015):
    rows = []
    for _ in range(n):
        is_fraud  = random.random() < fraud_rate
        merchant  = random.choice(MERCHANTS)
        country   = "GB" if not is_fraud else random.choice(COUNTRIES)
        currency  = "GBP" if country == "GB" else random.choice(CURRENCIES)
        amount    = (
            round(random.uniform(800,  9500), 2) if is_fraud
            else round(random.uniform(1.50, 350), 2)
        )
        rows.append(Row(
            txn_id          = str(uuid.uuid4()),
            account_id      = f"ACC{random.randint(1000,9999)}",
            customer_id     = f"CUST{random.randint(100,999)}",
            counterparty_id = f"CP{random.randint(1,200)}" if random.random() > 0.4 else None,
            txn_type        = random.choice(TXN_TYPES),
            channel         = random.choice(CHANNELS),
            amount          = amount,
            currency        = currency,
            merchant_name   = merchant[0],
            merchant_mcc    = merchant[1],
            country_code    = country,
            is_international= country != "GB",
            device_id       = f"DEV{random.randint(1,5000)}" if random.random() > 0.3 else None,
            ip_address      = f"10.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
            event_ts        = datetime.now(timezone.utc),
            kafka_offset    = random.randint(0, 10_000_000),
            kafka_partition = random.randint(0, 11),
        ))
    return rows

spark.createDataFrame(
    generate_batch(n=50_000, fraud_rate=0.015),
    schema=TXN_SCHEMA
).write.format("delta").mode("append") \
 .saveAsTable("banking_demo.bronze.transactions_source")